# Hello World — PySpark

Vérifie que l'environnement `uv` + PySpark + Jupyter fonctionne.

```bash
uv run jupyter lab
```

In [ ]:
import sys
print(f"Python {sys.version}")

In [ ]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("BGES - Hello World")
    .master("local[*]")
    .config("spark.driver.host", "localhost")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"PySpark {spark.version}")

## DataFrame de test

In [ ]:
data = [
    ("Paris",       "France",   2_161_000),
    ("Berlin",      "Germany",  3_645_000),
    ("London",      "UK",       8_982_000),
    ("New York",    "USA",      8_336_000),
    ("Los Angeles", "USA",      3_979_000),
    ("Shanghai",    "China",   24_870_000),
]

sdf = spark.createDataFrame(data, ["city", "country", "population"])
sdf.printSchema()
sdf.show()

In [ ]:
sdf.groupBy("country") \
   .agg(F.sum("population").alias("total_population")) \
   .orderBy(F.desc("total_population")) \
   .show()

## Visualisation

In [ ]:
import matplotlib.pyplot as plt

pdf = sdf.toPandas().sort_values("population", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(pdf["city"], pdf["population"] / 1_000_000, color="steelblue")
ax.set_xlabel("Population (millions)")
ax.set_title("Villes du projet BGES")
plt.tight_layout()
plt.show()

In [ ]:
spark.stop()